# Statistiques descriptives du segment « Mono-détenus sans engagement »

Dans ce notebook, nous décrivons la population du segment SA avant de construire le score. Nous cherchons à répondre à quatre questions :

1. **Qui sont les clients du segment**, et comment le risque varie-t-il selon leur profil (âge, ancienneté, CSP, mois d'observation) ?
2. **Les données sont-elles de bonne qualité ?** (valeurs manquantes, variables constantes)
3. **Quelles variables sont liées au défaut**, et lesquelles semblent les plus discriminantes ?
4. **Quelles variables disent la même chose** (redondances) ?

| Partie | Contenu |
|---|---|
| 0 | Paramètres, chargement et style des graphiques |
| 1 | Vue d'ensemble : volume et taux de défaut par mois |
| 2 | Profils de risque selon les caractéristiques des clients |
| 3 | Qualité des données : valeurs manquantes et variables constantes |
| 4 | Lien de chaque variable avec le défaut : V de Cramér, Kruskal-Wallis, IV |
| 5 | Fiches des variables les plus discriminantes |
| 6 | Distributions des clients sains et des clients en défaut |
| 7 | Redondances entre variables |
| 8 | Synthèse |

Ces statistiques sont calculées sur **toute la base du segment**. Elles servent seulement à décrire les données : les choix de modélisation sont faits dans le notebook d'analyse, sur l'échantillon d'apprentissage uniquement.

## 0. Paramètres et chargement

In [ ]:
# Librairies
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display
from scipy import stats

from utils import binning, exploration

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(80)

In [ ]:
# Colonnes principales
ID, TARGET, DATE, SEGMENT = "gdt", "ddefaut_ndb", "datdelhis", "segment"
BASE = Path(os.environ.get("SCORING_BASE", "data/segments/mono_sans_engagement/base_mono_sans_engagement.parquet"))

# Colonnes qui ne sont pas des variables explicatives
EXCLURE = {ID, TARGET, DATE, SEGMENT, "has_engagement", "SelectionProb", "SamplingWeight",
           "type_PM_PP", "code_marche_v2"}

# Nombre de variables affichées dans les graphiques de classement
TOP_N = 30

In [ ]:
# Lecture de la base ; les NaN deviennent des valeurs manquantes
base = pl.read_parquet(BASE)
base = exploration.nan_to_null(base)

# Répartition globale du segment
base.group_by(SEGMENT).agg(
    n=pl.len(),
    n_defaut=pl.col(TARGET).sum(),
    taux_defaut=pl.col(TARGET).mean(),
).sort(SEGMENT)

Le segment SA_mono_sans_eng comprend 78 118 observations, dont 449 défauts. Son taux de défaut est de 0,5748 %, ce qui signifie qu’environ 0,57 % des observations de ce segment sont en défaut, soit environ 6 défauts pour 1 000 observations.

In [ ]:
# Listes de variables
VARS = [c for c, t in base.schema.items() if c not in EXCLURE and not t.is_temporal()]
VARS_NUM = [c for c in VARS if base.schema[c].is_numeric()]
VARS_QUALI = [c for c in VARS if c not in VARS_NUM]

TAUX_MOYEN = base[TARGET].mean()
print(f"{len(VARS)} variables analysées : {len(VARS_NUM)} numériques et {len(VARS_QUALI)} qualitatives")
print(f"Taux de défaut moyen du segment : {TAUX_MOYEN:.3%}")

### Style des graphiques

Tous les graphiques utilisent les mêmes couleurs :

- **bleu** pour la série principale (les clients sains, l'apprentissage, une liaison significative) ;
- **orange** pour la série comparée (les clients en défaut, une moyenne de référence) ;
- **gris** pour ce qui n'est pas significatif ;
- une **gamme de bleus**, du plus clair au plus foncé, quand les catégories sont ordonnées (classes d'une variable, niveaux de pouvoir prédictif).

Nous n'utilisons jamais deux échelles verticales sur un même graphique : quand deux mesures n'ont pas la même unité, nous les plaçons côte à côte.

In [ ]:
COULEURS = {
    "bleu": "#2a78d6",
    "orange": "#eb6834",
    "vert": "#1baf7a",
    "gris": "#b4b2a9",
    "texte": "#0b0b0b",
    "texte_secondaire": "#52514e",
    "grille": "#e4e2dc",
    "fond": "#fcfcfb",
}
# Gamme de bleus, du plus clair au plus foncé, pour les catégories ordonnées
GAMME_BLEUE = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]

In [ ]:
def appliquer_style():
    # Style commun : fond clair, grille discrète, pas de cadre en haut et à droite
    plt.rcParams.update({
        "figure.facecolor": COULEURS["fond"],
        "axes.facecolor": COULEURS["fond"],
        "axes.edgecolor": COULEURS["grille"],
        "axes.labelcolor": COULEURS["texte_secondaire"],
        "axes.titlecolor": COULEURS["texte"],
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": COULEURS["grille"],
        "grid.linewidth": 0.6,
        "xtick.color": COULEURS["texte_secondaire"],
        "ytick.color": COULEURS["texte_secondaire"],
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.frameon": False,
        "legend.fontsize": 8,
        "lines.linewidth": 2,
        "lines.markersize": 6,
        "font.size": 9,
    })


appliquer_style()

## 1. Vue d'ensemble : volume et taux de défaut par mois

Nous regardons d'abord comment les observations et les défauts se répartissent sur les quatre mois d'observation.

In [ ]:
def resume_par_mois(df):
    # Nombre d'observations, de défauts et taux de défaut pour chaque mois
    return (
        df.group_by(DATE)
        .agg(n=pl.len(), n_defaut=pl.col(TARGET).sum(), taux_defaut=pl.col(TARGET).mean())
        .sort(DATE)
    )

In [ ]:
def graphique_par_mois(table):
    # À gauche le volume, à droite le taux de défaut (deux graphiques, pas de double axe)
    mois = [str(m) for m in table[DATE].to_list()]
    fig, (ax_volume, ax_taux) = plt.subplots(1, 2, figsize=(12, 3.8))

    ax_volume.bar(mois, table["n"], color=COULEURS["bleu"], width=0.6)
    ax_volume.set_title("Nombre d'observations par mois")
    for x, n in zip(mois, table["n"]):
        ax_volume.annotate(f"{n:,}".replace(",", " "), (x, n), ha="center", va="bottom", fontsize=8,
                           color=COULEURS["texte_secondaire"])

    taux = table["taux_defaut"] * 100
    ax_taux.plot(mois, taux, marker="o", color=COULEURS["bleu"])
    ax_taux.axhline(TAUX_MOYEN * 100, color=COULEURS["orange"], linestyle="--", linewidth=1.2)
    ax_taux.set_title(f"Taux de défaut par mois (%) - pointillés : moyenne ({TAUX_MOYEN:.2%})")
    ax_taux.set_ylim(0, taux.max() * 1.3)
    for x, t in zip(mois, taux):
        ax_taux.annotate(f"{t:.2f} %", (x, t), textcoords="offset points", xytext=(0, 8), ha="center",
                         fontsize=8, color=COULEURS["texte_secondaire"])
    fig.tight_layout()
    plt.show()

In [ ]:
par_mois = resume_par_mois(base)
display(par_mois)
graphique_par_mois(par_mois)

# Le taux de défaut est-il le même tous les mois ? (test du chi²)
tableau_mois = par_mois.select((pl.col("n") - pl.col("n_defaut")).alias("sains"), "n_defaut").to_numpy()
chi2_mois, p_mois, _, _ = stats.chi2_contingency(tableau_mois)
print(f"Test du chi² d'homogénéité entre les mois : chi² = {chi2_mois:.2f}, p-value = {p_mois:.4f}")

**Interprétation.** Les volumes sont stables d'un mois à l'autre : environ 19 000 à 20 000 observations par mois. Le taux de défaut baisse légèrement de janvier à mars (de 0,55 % à 0,44 %), puis remonte nettement en avril (0,82 %). Le test du chi² indique si cet écart est significatif : *à compléter avec la p-value*.

Cette variation dans le temps est importante pour la suite. Nous devrons vérifier que les variables et les classes de risque restent stables d'un mois à l'autre, et nous ne pourrons pas utiliser directement le taux d'un seul mois comme probabilité de défaut.

## 2. Profils de risque selon les caractéristiques des clients

Nous comparons maintenant le risque selon quelques caractéristiques simples des clients : l'âge, l'ancienneté de la relation, la catégorie socioprofessionnelle (CSP) et le type de clientèle (réseau ou Banque Privée).

Pour chaque caractéristique, nous affichons côte à côte :

- **la part des observations** dans chaque groupe, pour savoir si le groupe est important ;
- **le taux de défaut** du groupe, comparé à la moyenne du segment (ligne orange).

In [ ]:
# Groupes socioprofessionnels (premier chiffre du code PCS de l'INSEE)
GROUPES_CSP = {
    "1": "Agriculteurs", "2": "Artisans, commerçants", "3": "Cadres", "4": "Prof. intermédiaires",
    "5": "Employés", "6": "Ouvriers", "7": "Retraités", "8": "Sans activité",
}
AXES_CLIENT = {"1": "Réseau", "2": "Banque Privée", "3": "Gestion de Fortune"}

# Ordre d'affichage des groupes (ordre naturel, et non alphabétique)
ORDRE_GROUPES = {
    "tranche_age": ["moins de 25 ans", "25-35 ans", "35-50 ans", "50-65 ans", "65 ans et plus", "Manquant"],
    "tranche_anciennete": ["moins d'1 an", "1-5 ans", "5-10 ans", "10-15 ans", "15 ans et plus", "Manquant"],
    "groupe_csp": list(GROUPES_CSP.values()) + ["Autre / manquant"],
    "type_clientele": list(AXES_CLIENT.values()) + ["Manquant"],
}

In [ ]:
def ajouter_tranches(df):
    # Crée les colonnes de profil utilisées dans les graphiques
    colonnes = []
    if "AGE_PP_EN_MOIS" in df.columns:
        colonnes.append(
            (pl.col("AGE_PP_EN_MOIS") / 12)
            .cut([25, 35, 50, 65], labels=["moins de 25 ans", "25-35 ans", "35-50 ans", "50-65 ans", "65 ans et plus"])
            .cast(pl.String).fill_null("Manquant").alias("tranche_age")
        )
    if "ANCIENNETE_modif" in df.columns:
        colonnes.append(
            pl.col("ANCIENNETE_modif")
            .cut([12, 60, 120, 180], labels=["moins d'1 an", "1-5 ans", "5-10 ans", "10-15 ans", "15 ans et plus"])
            .cast(pl.String).fill_null("Manquant").alias("tranche_anciennete")
        )
    if "CODACVPRO_modif" in df.columns:
        colonnes.append(
            pl.col("CODACVPRO_modif").cast(pl.Int64, strict=False).cast(pl.String).str.slice(0, 1)
            .replace_strict(GROUPES_CSP, default="Autre / manquant").alias("groupe_csp")
        )
    if "code_axe_unite" in df.columns:
        colonnes.append(
            pl.col("code_axe_unite").cast(pl.Int64, strict=False).cast(pl.String)
            .replace_strict(AXES_CLIENT, default="Manquant").alias("type_clientele")
        )
    colonnes.append(pl.col(DATE).cast(pl.String).alias("mois_observation"))
    return df.with_columns(colonnes)

In [ ]:
def profil_risque(df, colonne):
    # Effectif, part de la population et taux de défaut de chaque groupe, dans l'ordre naturel des groupes
    table = (
        df.group_by(colonne)
        .agg(n=pl.len(), n_defaut=pl.col(TARGET).sum(), taux_defaut=pl.col(TARGET).mean())
        .with_columns(part=pl.col("n") / df.height)
    )
    ordre = ORDRE_GROUPES.get(colonne)
    if ordre is None:
        return table.sort(colonne)
    rang = {g: i for i, g in enumerate(ordre)}
    return (table.with_columns(rang=pl.col(colonne).replace_strict(rang, default=len(ordre)))
            .sort("rang").drop("rang"))

In [ ]:
def graphique_profil(table, colonne, titre):
    # Deux barres horizontales côte à côte : part des observations et taux de défaut
    groupes = table[colonne].to_list()
    y = np.arange(len(groupes))
    fig, (ax_part, ax_taux) = plt.subplots(1, 2, figsize=(12, 0.45 * len(groupes) + 1.4), sharey=True)
    fig.suptitle(titre, x=0.01, ha="left", fontsize=12, fontweight="bold", color=COULEURS["texte"])

    ax_part.barh(y, table["part"] * 100, color=COULEURS["bleu"], height=0.6)
    ax_part.set_title("Part des observations (%)")
    ax_part.set_yticks(y, groupes)
    ax_part.invert_yaxis()
    for yi, p in zip(y, table["part"] * 100):
        ax_part.annotate(f"{p:.1f} %", (p, yi), xytext=(3, 0), textcoords="offset points", va="center", fontsize=8,
                         color=COULEURS["texte_secondaire"])

    ax_taux.barh(y, table["taux_defaut"] * 100, color=COULEURS["bleu"], height=0.6)
    ax_taux.axvline(TAUX_MOYEN * 100, color=COULEURS["orange"], linestyle="--", linewidth=1.2)
    ax_taux.set_title(f"Taux de défaut (%) - pointillés : moyenne du segment ({TAUX_MOYEN:.2%})")
    ax_taux.set_xlim(0, table["taux_defaut"].max() * 100 * 1.45)
    for yi, t, d in zip(y, table["taux_defaut"] * 100, table["n_defaut"]):
        ax_taux.annotate(f"{t:.2f} % ({d} déf.)", (t, yi), xytext=(3, 0), textcoords="offset points",
                         va="center", fontsize=8, color=COULEURS["texte_secondaire"])
    ax_part.set_xlim(0, table["part"].max() * 100 * 1.2)
    ax_part.grid(axis="y", visible=False)
    ax_taux.grid(axis="y", visible=False)
    fig.tight_layout()
    plt.show()

In [ ]:
base_profils = ajouter_tranches(base)

PROFILS = {
    "tranche_age": "Âge du client",
    "tranche_anciennete": "Ancienneté de la relation",
    "groupe_csp": "Catégorie socioprofessionnelle",
    "type_clientele": "Type de clientèle",
}

for colonne, titre in PROFILS.items():
    if colonne in base_profils.columns:
        table = profil_risque(base_profils, colonne)
        display(table)
        graphique_profil(table, colonne, titre)

**Interprétation :** à compléter. Pour chaque caractéristique :

- le risque varie-t-il de façon régulière (par exemple, baisse-t-il avec l'âge ou l'ancienneté) ?
- quels groupes sont nettement au-dessus de la moyenne ?
- ces groupes sont-ils assez grands pour peser dans le modèle ? Un groupe très risqué mais avec très peu de défauts reste fragile.

Dans la segmentation, nous avions observé que le risque baisse avec l'ancienneté de la relation et qu'il est plus élevé chez les 25-40 ans. Nous vérifions ici si c'est aussi le cas dans le segment SA.

## 3. Qualité des données

### 3.1 Valeurs manquantes

Une variable presque toujours manquante n'apporte pas d'information. Nous calculons le taux de valeurs manquantes de chaque variable et nous affichons les plus touchées. La ligne orange marque le seuil de 90 % utilisé ensuite pour retirer une variable.

In [ ]:
def taux_manquants(df, variables):
    # Taux de valeurs manquantes de chaque variable, du plus élevé au plus faible
    return pl.DataFrame({
        "variable": variables,
        "pct_manquant": [df[c].null_count() / df.height for c in variables],
    }).sort("pct_manquant", descending=True)

In [ ]:
def graphique_barres(table, colonne_valeur, titre, couleurs=None, seuil=None, libelle_seuil=None,
                     echelle_log=False, format_valeur="{:.2f}", largeur=10):
    # Barres horizontales triées, avec la valeur écrite au bout de chaque barre
    noms = table["variable"].to_list()
    valeurs = table[colonne_valeur].to_numpy()
    y = np.arange(len(noms))
    fig, ax = plt.subplots(figsize=(largeur, 0.28 * len(noms) + 1.2))
    ax.barh(y, valeurs, color=couleurs if couleurs is not None else COULEURS["bleu"], height=0.65)
    ax.set_yticks(y, noms, fontsize=7)
    ax.invert_yaxis()
    ax.set_title(titre)
    ax.grid(axis="y", visible=False)
    if echelle_log:
        ax.set_xscale("log")
    if seuil is not None:
        ax.axvline(seuil, color=COULEURS["orange"], linestyle="--", linewidth=1.2, label=libelle_seuil)
        ax.legend(loc="lower right")
    for yi, v in zip(y, valeurs):
        ax.annotate(format_valeur.format(v), (v, yi), xytext=(3, 0), textcoords="offset points", va="center",
                    fontsize=7, color=COULEURS["texte_secondaire"])
    fig.tight_layout()
    plt.show()

In [ ]:
stats_manquants = taux_manquants(base, VARS)
print("Les 10 variables les plus manquantes :")
print(stats_manquants.head(10))

graphique_barres(
    stats_manquants.head(TOP_N).with_columns(pct=pl.col("pct_manquant") * 100),
    "pct", f"Les {TOP_N} variables les plus souvent manquantes (%)",
    seuil=90, libelle_seuil="seuil de retrait (90 %)", format_valeur="{:.1f} %",
)

In [ ]:
# Statistiques des variables numériques (moyenne, écart-type, quartiles...)
base.select(VARS_NUM).describe()

L'analyse des valeurs manquantes met en évidence une forte concentration de données absentes sur certaines variables. Parmi les dix variables présentant les taux de valeurs manquantes les plus élevés, cinq sont entièrement manquantes : `AGE_PM_EN_MOIS`, `LISTE_CODACVPRO_modif`, `LISTE_CODNAF2`, `LISTE_CODETAJUR` et `RATIO_LITIGES_vs_CA`, avec un taux de valeurs manquantes de 100 %. Ces variables ne contiennent donc aucune information exploitable directement dans l'échantillon étudié.

Les cinq autres variables présentent également des taux de valeurs manquantes très élevés, compris entre 98,76 % et 98,87 %. En particulier, `MVTAFF_STABILITE_2T_vs_12M`, `CODITDBDF_PRO`, `CRTOC_TOK_0010`, `flux_pro` et `CODNAF2` ne disposent que d'une faible proportion de valeurs renseignées.

**Implications pour la modélisation :**

* **Exclusion potentielle :** les variables entièrement manquantes doivent être exclues du modèle en l'état, puisqu'elles ne permettent ni de distinguer les observations ni d'estimer une relation avec la variable cible. Leur définition et leur disponibilité doivent néanmoins être vérifiées avant une suppression définitive.
* **Faible apport informationnel potentiel :** les variables présentant près de 99 % de valeurs manquantes risquent d'apporter peu d'information exploitable et de rendre les estimations instables si elles sont conservées sans traitement adapté.
* **Recherche des causes :** il convient de vérifier si ces absences résultent d'un problème d'extraction, d'une indisponibilité des données ou de caractéristiques structurelles du portefeuille. Une variable absente uniquement pour certains segments de clientèle peut, par exemple, refléter une différence de couverture des données.
* **Prévention des biais :** une imputation systématique n'est pas recommandée à ce stade. Elle pourrait introduire une information artificielle et masquer les problèmes de qualité des données.

En conclusion, ces résultats justifient un premier filtrage des variables selon leur taux de valeurs manquantes, suivi d'une analyse métier des variables les plus incomplètes. Le seuil d'exclusion devra être défini en fonction de la qualité des données, de leur pertinence pour le risque de crédit et de leur disponibilité en conditions réelles d'utilisation du modèle. Les décisions de traitement devront être prises sur l'échantillon d'entraînement, puis appliquées sans réestimation aux échantillons de validation et de test.


### 3.2 Variables constantes ou presque

Une variable dont une seule valeur couvre 99 % des lignes ou plus ne distingue presque aucun client. Nous séparons les variables **strictement constantes** (une seule valeur) des variables **quasi constantes**.

In [ ]:
const = exploration.constant_report(base, VARS, seuil=0.99)

constantes_strictes = const.filter(pl.col("n_valeurs") == 1)
quasi_constantes = const.filter((pl.col("part_top") >= 0.99) & (pl.col("n_valeurs") > 1))

print("Variables strictement constantes :", constantes_strictes.height)
print("Variables quasi constantes :", quasi_constantes.height)
print(quasi_constantes.select("variable", "n_valeurs", "valeur_top", "part_top").sort("part_top", descending=True))

Après exclusion des variables strictement constantes, 15 variables présentent une proportion de leur modalité dominante supérieure ou égale à 99 %. La modalité la plus fréquente est zéro pour 14 d'entre elles, tandis que `CODITDBDF_PAR` est dominée par la valeur `"0000"`. Cette concentration traduit une faible variabilité des variables sur l'échantillon d'entraînement.

Les cas les plus marqués sont `CRTAD_IND_0062`, `TX_UTIL_CCT_PRO` et `CRTOC_IND_0301`, dont la modalité dominante représente respectivement 99,96 %, 99,94 % et 99,93 % des observations. D'autres variables, comme `CRTOC_IND_0010` et `CRTOC_MOY_MVTAFF_IND_0010`, présentent une diversité importante de valeurs distinctes, mais restent dominées à 99,01 % par la valeur zéro. Le nombre de modalités ne suffit donc pas, à lui seul, à caractériser leur variabilité effective.

**Implications pour la modélisation :**

* **Pouvoir discriminant potentiellement limité :** la faible fréquence des modalités alternatives peut réduire la capacité de ces variables à distinguer les emprunteurs selon leur risque de défaut.
* **Risque d'estimations instables :** dans une régression logistique, les modalités rares peuvent entraîner des coefficients peu précis, voire des problèmes de séparation lorsque les événements de défaut sont très peu nombreux.
* **Vérification de la pertinence métier :** certaines variables peuvent néanmoins capturer des situations spécifiques, comme un incident de paiement ou une utilisation particulière du crédit. Leur faible fréquence ne justifie donc pas automatiquement leur suppression.
* **Sélection raisonnée :** la décision de conserver ou d'exclure ces variables doit tenir compte du nombre d'observations et de défauts associés aux modalités rares, de leur association avec la cible et de leur stabilité sur les échantillons de validation et de test.

En conclusion, ces 15 variables sont candidates à une sélection complémentaire, mais ne doivent pas être supprimées automatiquement sur la seule base du seuil de 99 %. Une analyse de leur distribution conditionnelle à la cible `ddefaut_ndb` permettra de mieux évaluer leur intérêt pour la prédiction du défaut.


## 4. Lien de chaque variable avec le défaut

Nous mesurons maintenant, variable par variable, si elle est liée au défaut. Nous utilisons trois mesures complémentaires.

| Mesure | Pour quelles variables | Ce qu'elle dit |
|---|---|---|
| **V de Cramér** (test du chi²) | variables qualitatives, et variables numériques avec 10 valeurs ou moins (indicateurs 0/1, compteurs) | la force du lien entre les catégories et le défaut, entre 0 et 1 |
| **Kruskal-Wallis** | variables numériques continues | si la variable prend des valeurs différentes chez les clients sains et chez les clients en défaut. Ce test fonctionne bien même quand les montants sont très dispersés |
| **Information Value (IV)** | toutes les variables, découpées en 10 classes | le pouvoir discriminant, avec les seuils de Siddiqi |

Dans les graphiques, une barre **bleue** signale un lien significatif à 5 %, une barre **grise** un lien non significatif.

Avec un taux de défaut d'environ 0,6 %, le V de Cramér reste toujours petit, même pour une variable très utile. Nous le lisons donc pour **classer** les variables entre elles, et non dans l'absolu.

In [ ]:
def liaison_qualitatives(df, variables, max_modalites=10):
    # Test du chi² et V de Cramér entre chaque variable « à peu de modalités » et le défaut
    lignes = []
    for v in variables:
        if df[v].n_unique() > max_modalites and df.schema[v].is_numeric():
            continue
        d = df.select(pl.col(v).cast(pl.String).fill_null("MANQUANT").alias(v), pl.col(TARGET))
        if d[v].n_unique() < 2:
            continue
        resultat = exploration.chi2_association(d, v, TARGET)
        resultat["n_modalites"] = d[v].n_unique()
        lignes.append(resultat)
    return pl.DataFrame(lignes).sort("cramers_v", descending=True)

In [ ]:
liaison_quali = liaison_qualitatives(base, VARS)
print(f"{liaison_quali.height} variables qualitatives ou à peu de modalités analysées")
display(liaison_quali.head(TOP_N))

couleurs = [COULEURS["bleu"] if p < 0.05 else COULEURS["gris"] for p in liaison_quali.head(TOP_N)["pvalue"]]
graphique_barres(liaison_quali.head(TOP_N), "cramers_v",
                 "Lien avec le défaut des variables qualitatives (V de Cramér)",
                 couleurs=couleurs, format_valeur="{:.3f}")

**Interprétation :** à compléter. Quelles variables qualitatives ou indicateurs sont les plus liés au défaut ? Nous nous attendons à retrouver en tête les indicateurs d'incidents (défaut ou arriéré passé, interdit de chéquier, solde négatif, litige).

In [ ]:
def liaison_quantitatives(df, variables, min_valeurs=11):
    # Test de Kruskal-Wallis : les valeurs diffèrent-elles entre clients sains et clients en défaut ?
    lignes = []
    for v in variables:
        d = df.select(pl.col(v).cast(pl.Float64).alias("x"), pl.col(TARGET).alias("y")).drop_nulls()
        if d["x"].n_unique() < min_valeurs or d["y"].n_unique() < 2:
            continue
        sains = d.filter(pl.col("y") == 0)["x"].to_numpy()
        defauts = d.filter(pl.col("y") == 1)["x"].to_numpy()
        h, p = stats.kruskal(sains, defauts)
        lignes.append({
            "variable": v, "H_kruskal": h, "p_value": p,
            "mediane_sains": float(np.median(sains)), "mediane_defauts": float(np.median(defauts)),
        })
    return pl.DataFrame(lignes).sort("H_kruskal", descending=True)

In [ ]:
liaison_quanti = liaison_quantitatives(base, VARS_NUM)
print(f"{liaison_quanti.height} variables quantitatives continues analysées")
display(liaison_quanti.head(TOP_N))

couleurs = [COULEURS["bleu"] if p < 0.05 else COULEURS["gris"] for p in liaison_quanti.head(TOP_N)["p_value"]]
graphique_barres(liaison_quanti.head(TOP_N), "H_kruskal",
                 "Lien avec le défaut des variables quantitatives (statistique H de Kruskal-Wallis, échelle log)",
                 couleurs=couleurs, echelle_log=True, format_valeur="{:.0f}")

**Interprétation :** à compléter. Quelles variables quantitatives séparent le mieux les clients sains des clients en défaut ? La comparaison des médianes (colonnes `mediane_sains` et `mediane_defauts`) indique dans quel sens joue chaque variable : par exemple, un solde médian plus faible chez les clients en défaut.

### 4.3 Pouvoir discriminant : l'Information Value (IV)

L'IV est calculée sur un découpage en 10 quantiles, avec une classe à part pour les valeurs manquantes. Nous la lisons avec les seuils de Siddiqi : en dessous de 0,02 la variable est inutile, de 0,02 à 0,1 faible, de 0,1 à 0,3 moyenne, de 0,3 à 0,5 forte, et au-dessus de 0,5 suspecte. Une IV au-dessus de 0,5 n'est pas forcément un problème : il faut surtout vérifier que la variable est bien connue avant la date d'observation.

Dans le graphique, la couleur suit le niveau de pouvoir prédictif : plus le bleu est foncé, plus la variable est discriminante. Les variables « suspectes » sont en orange.

In [ ]:
ivs = binning.iv_ranking(base, VARS, TARGET)

# Nombre de variables par niveau de pouvoir prédictif
print(ivs.group_by("pouvoir").len().sort("len", descending=True))
print(ivs.sort("iv", descending=True).head(TOP_N))

In [ ]:
COULEUR_POUVOIR = {
    "inutile": GAMME_BLEUE[0],
    "faible": GAMME_BLEUE[1],
    "moyen": GAMME_BLEUE[2],
    "fort": GAMME_BLEUE[4],
    "suspect": COULEURS["orange"],
}

top_iv = ivs.sort("iv", descending=True).head(TOP_N)
couleurs = [COULEUR_POUVOIR[p] for p in top_iv["pouvoir"]]
graphique_barres(top_iv, "iv", f"Les {TOP_N} variables de plus forte IV (seuil « inutile » à 0,02)",
                 couleurs=couleurs, seuil=0.02, libelle_seuil="seuil de 0,02", format_valeur="{:.2f}")


L'analyse de l'Information Value (IV) met en évidence une hétérogénéité du pouvoir discriminant des variables explicatives. Sur les 175 variables analysées, 87 sont classées comme inutiles, 27 présentent un pouvoir discriminant faible, 26 un pouvoir moyen, 16 un pouvoir fort et 19 sont classées comme suspectes.

Les variables jugées inutiles représentent près de la moitié des variables étudiées (49,7 %). Elles apportent peu d'information pour distinguer les emprunteurs défaillants des emprunteurs non défaillants. À l'inverse, 69 variables sont classées dans les catégories faible, moyen ou fort, soit 39,4 % du total. Les 19 variables suspectes représentent 10,9 % des variables analysées et nécessitent une vérification particulière.

**Implications pour la modélisation :**

* **Réduction de la dimension :** les variables classées comme inutiles constituent des candidates à l'exclusion, sous réserve de vérifier les seuils utilisés et leur éventuel intérêt métier.
* **Présélection des variables :** les variables présentant un IV moyen ou fort peuvent être examinées en priorité pour construire le modèle, sans être retenues automatiquement.
* **Vigilance sur les variables suspectes :** un IV anormalement élevé peut révéler une fuite de cible, une variable construite à partir d'informations postérieures au défaut ou une séparation excessive liée à des modalités rares. Ces variables doivent être auditées avant toute utilisation.
* **Validation de la stabilité :** le pouvoir discriminant observé sur l'échantillon d'entraînement doit être confirmé sur la validation et, idéalement, sur un échantillon temporel hors période d'estimation.

En conclusion, l'analyse IV constitue une première étape de sélection univariée. Elle doit être complétée par l'analyse des valeurs manquantes, des variables constantes, de la redondance entre variables et de la stabilité temporelle afin de construire un modèle de scoring robuste.


In [ ]:
# Variables avec une IV supérieure à 0,5 (à contrôler) et inférieure à 0,02 (inutiles)
print("IV supérieure à 0,5 :")
print(ivs.filter(pl.col("iv") > 0.5).sort("iv", descending=True))
print("\nIV inférieure à 0,02 :")
print(ivs.filter(pl.col("iv") < 0.02).sort("iv", descending=True))

## 5. Fiches des variables les plus discriminantes

Pour les variables de plus forte IV, nous construisons une fiche qui résume en un coup d'œil leur comportement. La variable est d'abord découpée en quelques classes avec les règles du cours (au moins 5 % des clients et 20 défauts par classe, au moins 30 % d'écart de risque entre deux classes voisines, chaque mois).

Chaque fiche contient quatre graphiques :

| Graphique | Ce qu'il montre | Ce que nous attendons |
|---|---|---|
| **Volume par mois** | la part des clients dans chaque classe, mois par mois | des courbes plates : la population est stable |
| **Risque par mois** | le taux de défaut de chaque classe, mois par mois | des courbes qui ne se croisent pas : l'ordre du risque est stable |
| **IV par classe** | la contribution de chaque classe à l'IV | savoir quelles classes portent l'information |
| **WoE par classe** | le Weight of Evidence de chaque classe | négatif = plus risqué que la moyenne, positif = moins risqué |

Les classes sont ordonnées, donc leurs couleurs vont du bleu clair (première classe) au bleu foncé (dernière classe).

In [ ]:
NB_FICHES = 6


def regle_descriptive(df, variable):
    # Découpage de la variable avec les règles du cours (quantiles pour une variable continue)
    continue_ = df.schema[variable].is_numeric() and df[variable].n_unique() > 10
    methode = "quantiles" if continue_ else "modalites"
    return binning.regle_par_methode(df, variable, TARGET, methode, DATE, n_bins=20,
                                     part_min=0.05, defauts_min=20, ecart_min=0.30)

In [ ]:
def classes_par_mois(df, variable, regle):
    # Part des observations et taux de défaut de chaque classe, pour chaque mois
    d = binning.appliquer_regle(df, variable, regle).with_columns(
        df[TARGET].alias("y"), df[DATE].cast(pl.String).alias("mois")
    )
    return (
        d.group_by("classe", "ordre", "mois")
        .agg(n=pl.len(), taux=pl.col("y").mean())
        .with_columns(part=pl.col("n") / pl.col("n").sum().over("mois"))
        .sort("ordre", "mois")
    )

In [ ]:
def graphique_fiche(df, variable, regle):
    # Fiche en quatre graphiques : volume par mois, risque par mois, IV par classe, WoE par classe
    table = binning.table_regle(df, variable, TARGET, regle)
    mensuel = classes_par_mois(df, variable, regle)
    classes = table["classe"].to_list()
    couleurs = {c: GAMME_BLEUE[min(i, len(GAMME_BLEUE) - 1)] for i, c in enumerate(classes)}
    if "missing" in couleurs:
        couleurs["missing"] = COULEURS["gris"]

    fig, axes = plt.subplots(2, 2, figsize=(13, 7))
    fig.suptitle(f"{variable}  |  IV = {table['iv'].sum():.3f}  |  {len(classes)} classes",
                 x=0.01, ha="left", fontsize=12, fontweight="bold", color=COULEURS["texte"])

    for c in classes:
        m = mensuel.filter(pl.col("classe") == c)
        axes[0, 0].plot(m["mois"], m["part"] * 100, marker="o", color=couleurs[c], label=c)
        axes[0, 1].plot(m["mois"], m["taux"] * 100, marker="o", color=couleurs[c], label=c)
    axes[0, 0].set_title("Volume : part des observations par mois (%)")
    axes[0, 1].set_title("Risque : taux de défaut par mois (%) - pointillés : moyenne")
    axes[0, 1].axhline(TAUX_MOYEN * 100, color=COULEURS["orange"], linestyle="--", linewidth=1.2)
    axes[0, 0].legend(fontsize=7, loc="best")

    y = np.arange(len(classes))
    axes[1, 0].barh(y, table["iv"], color=[couleurs[c] for c in classes], height=0.6)
    axes[1, 0].set_title("IV par classe")
    axes[1, 1].barh(y, table["woe"], color=[couleurs[c] for c in classes], height=0.6)
    axes[1, 1].axvline(0, color=COULEURS["texte_secondaire"], linewidth=0.8)
    axes[1, 1].set_title("WoE par classe (négatif = plus risqué que la moyenne)")

    for ax in (axes[1, 0], axes[1, 1]):
        ax.set_yticks(y, classes, fontsize=7)
        ax.invert_yaxis()
        ax.grid(axis="y", visible=False)
    for yi, iv, t, n, d in zip(y, table["iv"], table["taux_defaut"], table["n"], table["n_defaut"]):
        axes[1, 0].annotate(f"IV {iv:.3f} | {t:.2%} | {n} obs. | {d} déf.", (iv, yi), xytext=(3, 0),
                            textcoords="offset points", va="center", fontsize=7, color=COULEURS["texte_secondaire"])
    axes[1, 0].set_xlim(0, table["iv"].max() * 2.2)
    limite = float(np.abs(table["woe"]).max()) * 1.5
    axes[1, 1].set_xlim(-limite, limite)
    for yi, w in zip(y, table["woe"]):
        axes[1, 1].annotate(f"{w:+.2f}", (w, yi), xytext=(3 if w >= 0 else -3, 0), textcoords="offset points",
                            va="center", ha="left" if w >= 0 else "right", fontsize=7,
                            color=COULEURS["texte_secondaire"])
    fig.tight_layout()
    plt.show()

In [ ]:
# Fiches des variables de plus forte IV (hors variables suspectes, contrôlées à part)
variables_fiches = ivs.filter(pl.col("pouvoir") != "suspect").sort("iv", descending=True)["variable"].head(NB_FICHES).to_list()

for v in variables_fiches:
    regle = regle_descriptive(base, v)
    graphique_fiche(base, v, regle)

**Interprétation :** à compléter pour chaque fiche.

- Les courbes de volume sont-elles plates ? Si oui, la répartition des clients est stable dans le temps.
- Les courbes de risque se croisent-elles ? Un croisement signifie qu'une classe n'est pas toujours plus risquée que sa voisine : le découpage est fragile.
- Quelle classe porte l'essentiel de l'IV ? C'est souvent une petite classe très risquée.
- Le WoE évolue-t-il de façon régulière d'une classe à l'autre ? C'est la monotonie attendue.

## 6. Distributions des clients sains et des clients en défaut

Pour les variables quantitatives les plus liées au défaut, nous superposons la distribution des clients sains (bleu) et celle des clients en défaut (orange). Plus les deux courbes sont décalées, plus la variable sépare bien les deux populations.

Les montants bancaires sont très dispersés. Nous les affichons donc sur une **échelle logarithmique signée** : sign(x) × log(1 + |x|). Cela resserre les grandes valeurs tout en gardant le signe (un solde négatif reste négatif).

In [ ]:
def log_signe(x):
    # Échelle logarithmique qui garde le signe de la valeur
    return np.sign(x) * np.log1p(np.abs(x))

In [ ]:
def graphique_distribution(df, variable, ax):
    # Distributions superposées des clients sains et des clients en défaut (densités normalisées)
    d = df.select(pl.col(variable).cast(pl.Float64).alias("x"), pl.col(TARGET).alias("y")).drop_nulls()
    x_sains = log_signe(d.filter(pl.col("y") == 0)["x"].to_numpy())
    x_defauts = log_signe(d.filter(pl.col("y") == 1)["x"].to_numpy())
    bornes = np.histogram_bin_edges(np.concatenate([x_sains, x_defauts]), bins=40)
    centres = (bornes[:-1] + bornes[1:]) / 2

    for x, couleur, nom in ((x_sains, COULEURS["bleu"], "sains"), (x_defauts, COULEURS["orange"], "en défaut")):
        densite, _ = np.histogram(x, bins=bornes, density=True)
        ax.plot(centres, densite, color=couleur, label=f"{nom} (n = {len(x):,})".replace(",", " "))
        ax.fill_between(centres, densite, color=couleur, alpha=0.15)

    ks = stats.ks_2samp(x_sains, x_defauts).statistic if len(x_defauts) > 1 else np.nan
    ax.set_title(f"{variable}  (KS = {ks:.2f})", fontsize=9)
    ax.set_xlabel("sign(x) × log(1 + |x|)")
    ax.legend(fontsize=7)

In [ ]:
variables_distribution = liaison_quanti["variable"].head(6).to_list()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for v, ax in zip(variables_distribution, axes.ravel()):
    graphique_distribution(base, v, ax)
for ax in axes.ravel()[len(variables_distribution):]:
    ax.set_visible(False)
fig.suptitle("Distributions des clients sains et des clients en défaut", x=0.01, ha="left",
             fontsize=12, fontweight="bold", color=COULEURS["texte"])
fig.tight_layout()
plt.show()

**Interprétation :** à compléter. Pour quelles variables les deux distributions sont-elles le plus décalées ? La statistique **KS** (écart maximal entre les deux répartitions) résume cet écart : plus elle est proche de 1, mieux la variable sépare les clients sains des clients en défaut.

## 7. Redondances entre variables

### 7.1 Paires de variables numériques très corrélées

Deux variables avec une corrélation de Spearman supérieure à 0,8 en valeur absolue disent presque la même chose. Dans le modèle, nous ne garderons que la plus utile des deux.

In [ ]:
VARS_NUM_CORR = [c for c in VARS if base.schema[c].is_numeric()]

paires = exploration.paires_redondantes(base, columns=VARS_NUM_CORR, seuil=0.8, n_max=50000, seed=0)
print("Nombre de paires redondantes :", paires.height)
print(paires.sort("spearman", descending=True))

Certaines variables présentent des redondances entre elles. Lors de la phase de modélisation, nous examinerons ces redondances afin de sélectionner les variables les plus pertinentes pour notre étude, en tenant compte notamment de leur disponibilité et de leur intérêt pour la prédiction du risque de défaut.


### 7.2 Carte des liens entre les variables les plus discriminantes

Pour voir d'un coup d'œil quelles variables se ressemblent, nous découpons les 30 variables de plus forte IV en déciles (plus une classe pour les valeurs manquantes), puis nous calculons le V de Cramér entre chaque paire. Une case foncée signale deux variables qui répartissent les clients presque de la même façon. Les valeurs sont écrites en pourcentage (100 = partition identique).

In [ ]:
def classes_deciles(df, variable):
    # Découpe une variable en déciles (ou garde ses modalités si elle en a peu) ; manquant = classe à part
    if df.schema[variable].is_numeric() and df[variable].n_unique() > 10:
        classe = pl.col(variable).cast(pl.Float64).qcut(10, allow_duplicates=True).cast(pl.String)
    else:
        classe = pl.col(variable).cast(pl.String)
    return df.select(classe.fill_null("MANQUANT").alias(variable))[variable]

In [ ]:
def matrice_cramer(df, variables):
    # V de Cramér entre toutes les paires de variables découpées
    classes = {v: classes_deciles(df, v) for v in variables}
    k = len(variables)
    m = np.full((k, k), np.nan)
    for i in range(k):
        for j in range(i):
            m[i, j] = binning._v_cramer(classes[variables[i]], classes[variables[j]])
    return m

In [ ]:
def graphique_heatmap(m, variables, titre):
    # Carte en triangle : plus la case est foncée, plus les deux variables se ressemblent
    from matplotlib.colors import LinearSegmentedColormap
    gamme = LinearSegmentedColormap.from_list("bleus", ["#f0efec", "#86b6ef", "#2a78d6", "#104281"])
    fig, ax = plt.subplots(figsize=(13, 11))
    image = ax.imshow(m * 100, cmap=gamme, vmin=0, vmax=100)
    ax.set_xticks(range(len(variables)), variables, rotation=90, fontsize=7)
    ax.set_yticks(range(len(variables)), variables, fontsize=7)
    ax.grid(False)
    for i in range(len(variables)):
        for j in range(i):
            valeur = m[i, j] * 100
            ax.text(j, i, f"{valeur:.0f}", ha="center", va="center", fontsize=6,
                    color="white" if valeur > 55 else COULEURS["texte_secondaire"])
    fig.colorbar(image, ax=ax, shrink=0.6, label="V de Cramér (%)")
    ax.set_title(titre)
    fig.tight_layout()
    plt.show()

In [ ]:
top_variables = ivs.sort("iv", descending=True)["variable"].head(TOP_N).to_list()
m_cramer = matrice_cramer(base, top_variables)
graphique_heatmap(m_cramer, top_variables, f"V de Cramér entre les {TOP_N} variables de plus forte IV")

**Interprétation :** à compléter. Quels groupes de variables se ressemblent fortement (cases foncées) ? Nous nous attendons à des blocs de variables qui mesurent la même chose sur des périodes différentes : soldes sur plusieurs trimestres, jours de découvert sur 3 et 15 mois, fenêtres d'arriérés voisines.

### 7.3 Liens entre variables qualitatives

Nous testons aussi, avec le chi² et le V de Cramér, si les variables qualitatives sont liées entre elles.

In [ ]:
def liens_qualitatives(df, variables):
    # Chi² et V de Cramér entre chaque paire de variables qualitatives
    lignes = []
    for i, a in enumerate(variables):
        for b in variables[i + 1:]:
            d = df.select(pl.col(a).cast(pl.String).fill_null("MANQUANT"), pl.col(b).cast(pl.String).fill_null("MANQUANT"))
            tableau = d.group_by(a, b).len().pivot(on=b, index=a, values="len").fill_null(0).drop(a).to_numpy()
            if min(tableau.shape) < 2:
                continue
            chi2, p, _, _ = stats.chi2_contingency(tableau, correction=False)
            v = np.sqrt(chi2 / (tableau.sum() * (min(tableau.shape) - 1)))
            lignes.append({"variable_1": a, "variable_2": b, "chi2": chi2, "p_value": p, "v_cramer": v})
    return pl.DataFrame(lignes).sort("v_cramer", descending=True)

In [ ]:
quali_a_tester = [v for v in VARS_QUALI if base[v].n_unique() > 1]
liens_quali = liens_qualitatives(base, quali_a_tester)
display(liens_quali)

L’analyse des associations entre variables qualitatives repose sur le test du Chi² d’indépendance et le V de Cramér. Le test du Chi² permet de déterminer si deux variables qualitatives sont statistiquement dépendantes, tandis que le V de Cramér mesure l’intensité de leur association, avec une valeur comprise entre 0 et 1.


Au seuil de significativité de 5 %, plusieurs associations statistiquement significatives sont observées, notamment entre `CODITDBDF_PAR` et `CODITDBDF_PRO`, entre `CODNAF2` et `CODETAJUR`, ainsi qu’entre `CAUSE_SAIN_M0` et `CODITDBDF_PAR`.

Ces résultats suggèrent que les distributions de certaines variables ne sont pas indépendantes. Toutefois, une p-value faible ne signifie pas nécessairement que l'association est forte. Il est donc nécessaire d'examiner le V de Cramér pour évaluer l'intensité des relations observées.

À l’inverse, certaines paires, comme `CODETAJUR` et `CRTOC_TOK_0010`, présentent une p-value supérieure à 5 %. Pour ces paires, l’hypothèse d’indépendance n’est pas rejetée au seuil retenu. Cela ne constitue cependant pas une preuve d’indépendance.

La variable `gdt` présente des associations statistiquement significatives avec plusieurs variables. Si elle correspond à un identifiant individuel ou à une clé technique, elle ne doit généralement pas être retenue comme variable explicative dans le modèle de scoring. Le nombre élevé de modalités peut également limiter l’interprétabilité du test du Chi².

Par ailleurs, lorsque les effectifs théoriques de certaines cellules du tableau de contingence sont trop faibles, l’approximation du Chi² peut être peu fiable. Il convient donc de vérifier les conditions d’application du test.

Enfin, la réalisation de nombreux tests augmente le risque de faux positifs. Une correction des p-values, telle que la procédure de Benjamini-Hochberg, peut être envisagée.


Pour la construction du modèle de scoring, cette analyse constitue une première étape d’identification des éventuelles redondances entre variables qualitatives. Les paires présentant un V de Cramér élevé doivent être examinées afin de déterminer si elles apportent une information similaire.

En cas de redondance avérée, le choix entre deux variables doit tenir compte de leur pertinence métier, de leur interprétabilité, de leur stabilité temporelle et de leur pouvoir prédictif. Une variable ne doit donc pas être supprimée uniquement parce que son association avec une autre variable est statistiquement significative.


Le test du Chi² met en évidence plusieurs dépendances statistiques entre variables qualitatives. Cependant, la significativité statistique ne suffit pas à caractériser l’importance de ces relations. L’examen du V de Cramér permet de mieux évaluer leur intensité et d’identifier les éventuelles redondances entre variables explicatives. Cette analyse doit être complétée par une évaluation métier et par l’étude de la stabilité et du pouvoir prédictif des variables avant leur sélection définitive.


### 7.4 Multicolinéarité : le VIF

Le VIF mesure à quel point une variable peut être retrouvée à partir des autres. Au-dessus de 10, la multicolinéarité est forte. Nous le calculons sur les variables numériques qui ont une IV d'au moins 0,02, en remplaçant les valeurs manquantes par la médiane pour ce calcul seulement.

In [ ]:
def calculer_vif(df, variables):
    # VIF de chaque variable : diagonale de l'inverse de la matrice de corrélation
    X = df.select([
        pl.col(v).cast(pl.Float64).fill_null(pl.col(v).cast(pl.Float64).median()).alias(v) for v in variables
    ]).to_numpy()
    non_constantes = X.std(axis=0) > 0
    X = X[:, non_constantes]
    noms = [v for v, garde in zip(variables, non_constantes) if garde]
    vif = np.diag(np.linalg.pinv(np.corrcoef(X, rowvar=False)))
    return pl.DataFrame({"variable": noms, "VIF": vif}).sort("VIF", descending=True)

In [ ]:
variables_vif = [v for v in ivs.filter(pl.col("iv") >= 0.02)["variable"].to_list() if v in VARS_NUM]
table_vif = calculer_vif(base, variables_vif)
display(table_vif.head(TOP_N))

couleurs = [COULEURS["orange"] if v > 10 else COULEURS["bleu"] for v in table_vif.head(TOP_N)["VIF"]]
graphique_barres(table_vif.head(TOP_N), "VIF", "VIF des variables numériques utiles (IV supérieure à 0,02)",
                 couleurs=couleurs, seuil=10, libelle_seuil="seuil de 10", echelle_log=True, format_valeur="{:.1f}")

**Interprétation :** à compléter. Combien de variables dépassent un VIF de 10 ? Ce sont en général les mêmes que celles repérées dans les paires redondantes et sur la carte du V de Cramér. Elles seront traitées au moment de la sélection des variables : nous ne garderons que la plus discriminante de chaque groupe.

## 8. Synthèse

*À compléter à partir des résultats ci-dessus.*

- **Population.** Le segment compte 78 118 observations et 449 défauts (0,57 %). Le taux de défaut varie dans le temps, avec une hausse nette en avril.
- **Profils de risque.** *Groupes les plus risqués selon l'âge, l'ancienneté et la CSP.*
- **Qualité des données.** *Nombre de variables presque vides ou quasi constantes, qui seront retirées.*
- **Variables les plus discriminantes.** *Les principales variables selon l'IV, le V de Cramér et Kruskal-Wallis, et leur sens.*
- **Redondances.** *Les principaux groupes de variables qui disent la même chose.*

Ces constats guident la préparation des données et la sélection des variables dans le notebook d'analyse (`Analyse_Mono_Sans_Engagement_decoupe.ipynb`).